In [ ]:
! pip install supermemory langchain langgraph python-dotenv langchain-mistralai langchain-groq langchain-core langchain-classic langchain-community langchain-mcp-adapters
! pip install rich

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import SystemMessage, HumanMessage
from supermemory import Supermemory
from dotenv import load_dotenv
import os
from rich import print
load_dotenv()

In [ ]:
mistrial_api = os.getenv("MISTRAL_API_KEY","")
memory_api = os.getenv("SUPERMEMORY_API_KEY","")


In [ ]:
llm = ChatMistralAI(model="mistral-small",api_key=mistrial_api)
memory = Supermemory(api_key=memory_api)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str

In [ ]:
def agent(state:State):
  user_id = state["user_id"]
  messages = state["messages"]
  user_query = messages[-1].content # latest content
  # Fetch user profile with relevant memories
  profile_result = memory.profile(container_tag=user_id, q=user_query) # so and similarity memory search

  # Build context from profile
  static_facts = profile_result.profile.static or [] # for initial message it is [] / static_facts : predefined not changing
  print(f"statis facts:\n{static_facts}")
  dynamic_context = profile_result.profile.dynamic or [] # dynamic facts : continuously updating memory like user taste
  print(f"dynamic facts:{dynamic_context}")
  search_results = profile_result.search_results.results if profile_result.search_results else [] # the total search result
  print(f"Search result:\n{search_results}")
  context = f"""
  User Background:
  {chr(10).join(static_facts) if static_facts else 'No profile yet.'}
  Recent Context:
  {chr(10).join(dynamic_context) if dynamic_context else 'No recent activity.'}
  Relevant Memories:
  {chr(10).join([r.memory or r.chunk for r in search_results]) if search_results else 'None found.'}
  """
  system = SystemMessage(content=f"You are a helpful assistant.\n\n{context}")
  response = llm.invoke([system] + messages)

  # Store the interaction
  memory.add(
      content=f"User: {user_query}\nAssistant: {response.content}",
      container_tag=user_id
  )

  return {"messages": [response]}

In [ ]:
# Build the graph
graph = StateGraph(State)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)
app = graph.compile()

In [ ]:
# Run it
result = app.invoke({
    "messages": [HumanMessage(content="Hi! I'm working on a Python project.")],
    "user_id": "user_123"
})
print(result["messages"][-1].content)

In [ ]:
while True :
  query = input("User:\n")
  if query in ("break","done","close"):
    break
  result = app.invoke({
      "messages":HumanMessage(content=query),
      "user_id":"user_123"
  })
  print(f"AI:\n{result["messages"][-1].content}") # no nedd to add back to messages
  print("==="*40)

 User:
my python project is chatbot with langgraph
AI:
That sounds like an exciting project! A chatbot with **LangGraph** (part of the LangChain ecosystem) can leverage
graph-based structures for more complex, stateful interactions.

### **Key Steps to Build Your Chatbot with LangGraph**
Here’s a high-level guide to get you started:

---

### **1. Install Required Libraries**
```bash
pip install langgraph langchain langchain-core
```
- **LangGraph**: For graph-based workflows.
- **LangChain**: For LLM integration and chains.
- **LangChain-Core**: Core utilities.

---

### **2. Define Your Chatbot’s Graph Structure**
LangGraph allows you to model interactions as a **state machine** or **graph**. Example:

```python
from langgraph.graph import Graph

# Define nodes (functions) and edges (transitions)
workflow = Graph()

def call_model(state):
    # Your LLM logic here (e.g., using LangChain's LLM)
    return {"response": "Hello! How can I help?"}

def should_continue(state):
    # Logic to decide if the conversation continues
    return "end"  # or "continue"

# Add nodes and edges
workflow.add_node("model", call_model)
workflow.add_node("should_continue", should_continue)

workflow.set_entry_point("model")
workflow.add_edge("model", "should_continue")
workflow.add_conditional_edges(
    "should_continue",
    lambda state: state["response"],  # Route based on response
    {"end": END, "continue": "model"}
)

# Compile the graph
app = workflow.compile()
```

---

### **3. Integrate an LLM (e.g., OpenAI, Ollama)**
Use **LangChain’s LLM wrappers** for responses:
```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4")

def call_model(state):
    response = llm.invoke([HumanMessage(content=state["input"])])
    return {"response": response.content}
```

---

### **4. Add Tools/Functions (Optional)**
For dynamic interactions, use **tools** (e.g., web search, APIs):
```python
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

def search_tool(state):
    return {"response": search.run(state["input"])}
```

---

### **5. Run the Chatbot**
```python
for event in app.stream({"input": "Hi!"}):
    print(event)
```

---

### **6. Advanced Features**
- **Memory**: Use `StateGraph` to track conversation history.
- **Conditional Logic**: Route based on user input (e.g., FAQ vs. open-ended).
- **Multi-Agent Systems**: Model interactions between multiple agents.

---

### **Example Project Structure**
```
chatbot/
├── main.py          # Graph workflow
├── llm.py           # LLM integration
├── tools.py         # Custom tools
└── memory.py        # State management
```

Would you like help with a specific part (e.g., memory, tools, or deployment)? 🚀

User:
that's great from your side i like to implement `supermemory` in it let`s start with it
AI:
Great! Implementing **SuperMemory** in your project sounds like an exciting idea. SuperMemory is a powerful concept
that can enhance your application's ability to retain and utilize information effectively.

### **How to Proceed?**
To implement SuperMemory, we need to clarify a few things:

1. **What is your project about?**
   - Is it a chatbot, a knowledge base, an AI assistant, or something else?
   - Does it already have some form of memory (e.g., a database, vector store, or in-memory cache)?

2. **How do you want SuperMemory to work?**
   - Should it remember user preferences, past interactions, or external knowledge?
   - Do you need long-term persistence (like a database) or short-term working memory?

3. **What tech stack are you using?**
   - Python, JavaScript, etc.?
   - Any specific frameworks (LangChain, LlamaIndex, etc.)?

### **Possible Implementation Approaches**
Depending on your needs, SuperMemory could be implemented via:
- **Vector Databases (Pinecone, Weaviate, FAISS)** – For semantic memory.
- **SQL/NoSQL Databases (PostgreSQL, MongoDB)** – For structured memory.
- **In-Memory Caching (Redis)** – For fast access.
- **Hybrid Approach (Combining multiple methods)** – Best for scalability.

### **Next Steps**
Could you provide more details about:
✅ Your project type?
✅ Desired memory features?
✅ Existing tech stack?

Once we have this, I can guide you through a step-by-step implementation! 🚀

User:
okay then what is vector db
AI:
A **Vector Database (Vector DB)** is a specialized type of database designed to store, index, and retrieve **vector
embeddings**—high-dimensional numerical representations of data (like text, images, or audio) generated by machine
learning models. These embeddings capture semantic meaning, enabling efficient similarity search and retrieval.

### **Key Features of Vector Databases:**
1. **Vector Embeddings**
   - Data (e.g., text, images) is converted into dense vectors (e.g., 384, 768, or 1024-dimensional arrays) using
models like:
     - **Text**: Sentence-BERT, Word2Vec, TF-IDF.
     - **Images**: CNNs (ResNet, ViT).
     - **Audio**: Wav2Vec, Whisper.
   - Example: The sentence *"The cat sat on the mat"* might be converted to a vector like `[0.2, -0.5, 0.8, ...]`.

2. **Similarity Search**
   - Finds vectors close to a query vector using metrics like:
     - **Cosine similarity** (angular distance).
     - **Euclidean distance** (straight-line distance).
     - **Dot product** (for normalized vectors).

3. **Optimized for High-Dimensional Data**
   - Traditional databases (SQL/NoSQL) struggle with high-dimensional vectors. Vector DBs use specialized indexing:
     - **ANN (Approximate Nearest Neighbor)** algorithms:
       - **HNSW** (Hierarchical Navigable Small World) – Fast and accurate.
       - **FAISS** (Facebook AI Similarity Search) – Optimized for GPU/CPU.
       - **ScaNN** (Google) – High-speed similarity search.
       - **PQ (Product Quantization)** – Compresses vectors for memory efficiency.

4. **Scalability & Performance**
   - Handles **millions/billions of vectors** efficiently.
   - Supports **real-time** or **batch** queries.

---

### **Why Use a Vector Database?**
| Use Case | Example |
|----------|---------|
| **Semantic Search** | Retrieve documents similar to a query (e.g., "Find research papers about climate change").
|
| **Recommendation Systems** | "Users who liked X also liked Y." |
| **Image/Video Search** | Find similar images (e.g., reverse image search). |
| **Chatbots & RAG** | Retrieve relevant context for LLMs (e.g., grounding answers in documents). |
| **Anomaly Detection** | Identify outliers (e.g., fraud detection). |
| **Multimodal Search** | Search across text, images, and audio (e.g., "Find images of red cars"). |

---

### **Popular Vector Databases**
| Database | Key Features | License | Use Case |
|----------|-------------|---------|----------|
| **Pinecone** | Managed, serverless, real-time indexing | Proprietary | Production apps, startups |
| **Weaviate** | Open-source, modular, supports GraphQL | BSD | AI-native apps, RAG |
| **Milvus / Zilliz** | Open-source, scalable, supports hybrid search | Apache 2.0 | Large-scale vector search |
| **Qdrant** | Open-source, lightweight, filtering support | Apache 2.0 | Edge deployments |
| **Chroma** | Open-source, simple, Python-first | Apache 2.0 | Prototyping, small projects |
| **FAISS** (Facebook) | Library (not a DB), optimized for ANN | MIT | Custom implementations |
| **Redis Stack** | In-memory, supports vectors + JSON | BSD | Low-latency caching |
| **Vespa** (Yahoo) | Combines search + AI, supports tensors | Apache 2.0 | Enterprise search |

---

### **How It Works (Example Workflow)**
1. **Ingest Data**:
   - Convert raw data (e.g., text) into vectors using an embedding model.
   - Store vectors in the Vector DB.
2. **Indexing**:
   - The DB builds an ANN index (e.g., HNSW) for fast similarity search.
3. **Querying**:
   - Convert a query (e.g., "What is AI?") into a vector.
   - Search for top-*k* similar vectors in the index.
   - Return the most relevant results.

---

### **Vector DB vs. Traditional Database**
| Feature | Vector DB | Traditional DB (SQL/NoSQL) |
|---------|-----------|----------------------------|
| **Data Type** | Vectors (floats) | Structured (rows/columns) |
| **Query Type** | Similarity search (ANN) | Exact matches (SQL) |
| **Scalability** | Optimized for high dimensions | Struggles with >100D vectors |
| **Use Case** | AI/ML, semantic search | Transactions, CRUD operations |

---

### **When to Use a Vector DB?**
✅ You need **semantic search** (not just keyword matching).
✅ Your data is **unstructured** (text, images, audio).
✅ You’re working with **AI/ML models** (LLMs, embeddings).
✅ You need **real-time or near-real-time** similarity search.

❌ Don’t use it for:
- Simple key-value lookups.
- Structured data with exact-match queries (use SQL/NoSQL instead).

---

### **Example Query (Pinecone)**
```python
import pinecone

# Initialize
pinecone.init(api_key="YOUR_API_KEY", environment="us-west1-gcp")
index = pinecone.Index("my-index")

# Query
query_vector = [0.1, 0.5, -0.2, ...]  # Embedded query
results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)
print(results)
```

---

### **Challenges**
1. **Embedding Quality**: Poor embeddings = poor search results.
2. **Indexing Time**: Building HNSW/FAISS indexes can be slow for large datasets.
3. **Cold Start**: New data requires re-indexing.
4. **Cost**: Managed services (Pinecone, Weaviate) can be expensive at scale.

---

### **Future Trends**
- **Hybrid Search**: Combining keyword + vector search (e.g., BM25 + ANN).
- **Multimodal Vectors**: Storing and searching across text, images, and audio.
- **On-Device Vector DBs**: Lightweight solutions for edge AI (e.g., TensorFlow Lite).
- **Real-Time Updates**: Streaming vector embeddings without full re-indexing.

Would you like a deeper dive into any specific aspect (e.g., how HNSW works, or a comparison between Pinecone and
Weaviate)?

User:
done
AI:
Got it! Let me know how I can assist you further. 😊

User:
break
